# Neural Networks Notebook

## 1. Setup & Data Loading

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Load  dataset
df = pd.read_csv('energydata_complete.csv')

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset loaded: 19735 rows, 29 columns


## 2. Exploratory Data Analysis & Preprocessing

In [12]:
print(f"Dataset shape: {df.shape}")
print(df.info())
print(df.describe())

# Convert date and extract time features
df['date'] = pd.to_datetime(df['date'])
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

# Feature correlation analysis
correlation_matrix = df.drop(columns=['date', 'rv1', 'rv2']).corr()
target_correlation = correlation_matrix['Appliances'].sort_values(ascending=False)

print("\nTop 10 features correlated with Appliances:")
print(target_correlation.head(10))

# Define features (X) and target (y)
X = df.drop(columns=['Appliances', 'date', 'rv1', 'rv2'])
y = df['Appliances']

# Train-test split (use shuffle=False if respecting time series order)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

# Standardise features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining samples: {X_train_scaled.shape[0]}, Features: {X_train_scaled.shape[1]}")
print(f"Target range: {y.min():.2f} to {y.max():.2f} Wh")
print(f"Target mean: {y.mean():.2f} Wh, std: {y.std():.2f} Wh")

Dataset shape: (19735, 29)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19735 entries, 0 to 19734
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         19735 non-null  object 
 1   Appliances   19735 non-null  int64  
 2   lights       19735 non-null  int64  
 3   T1           19735 non-null  float64
 4   RH_1         19735 non-null  float64
 5   T2           19735 non-null  float64
 6   RH_2         19735 non-null  float64
 7   T3           19735 non-null  float64
 8   RH_3         19735 non-null  float64
 9   T4           19735 non-null  float64
 10  RH_4         19735 non-null  float64
 11  T5           19735 non-null  float64
 12  RH_5         19735 non-null  float64
 13  T6           19735 non-null  float64
 14  RH_6         19735 non-null  float64
 15  T7           19735 non-null  float64
 16  RH_7         19735 non-null  float64
 17  T8           19735 non-null  float64
 18  RH_8         19735 

## 3. Baseline Neural Network Model

In [13]:
def create_baseline_model(input_dim):
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)  # Output layer for regression
    ])
    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )
    return model

baseline_model = create_baseline_model(X_train_scaled.shape[1])
baseline_model.summary()

# Train the baseline model
print("\nTraining baseline model...")
history_baseline = baseline_model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

# Evaluate baseline
y_pred_baseline = baseline_model.predict(X_test_scaled).flatten()
test_rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
test_r2_baseline = r2_score(y_test, y_pred_baseline)
test_mae_baseline = np.mean(np.abs(y_test - y_pred_baseline))
test_mape_baseline = np.mean(np.abs((y_test - y_pred_baseline) / y_test)) * 100

print(f"\n--- Baseline Model Results ---")
print(f"Test RMSE: {test_rmse_baseline:.2f} Wh")
print(f"Test MAE: {test_mae_baseline:.2f} Wh")
print(f"Test MAPE: {test_mape_baseline:.2f}%")
print(f"Test R²: {test_r2_baseline:.3f}")
print(f"MAE/Mean Ratio: {test_mae_baseline/y_test.mean():.2%}")

c:\Users\jenso\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_54 (Dense)                │ (None, 64)             │         1,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_55 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_56 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,969 (15.50 KB)

 Trainable params: 3,969 (15.50 KB)

 Non-trainable params: 0 (0.00 B)


Training baseline model...
Epoch 1/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 13109.1729 - mae: 65.9097 - val_loss: 9558.5088 - val_mae: 56.6346
Epoch 2/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 898us/step - loss: 9870.8418 - mae: 55.4335 - val_loss: 8717.8291 - val_mae: 53.2412
Epoch 3/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 883us/step - loss: 9364.6396 - mae: 53.7597 - val_loss: 8362.2930 - val_mae: 52.1178
Epoch 4/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 938us/step - loss: 9108.7363 - mae: 53.0506 - val_loss: 8175.5532 - val_mae: 51.5672
Epoch 5/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 892us/step - loss: 8950.2871 - mae: 52.5606 - val_loss: 8062.1978 - val_mae: 51.1712
Epoch 6/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 901us/step - loss: 8840.0801 - mae: 52.1388 - val_loss: 7980.9019 - val_mae: 50.9481
Epoch 7/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 938us/step - loss: 8753.0898 - mae: 51.7871 - val_loss: 7910.3252 - val_mae: 50.6241
Epoch 8/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 882us/step - loss: 8677.7383 - mae

## 4. Hyperparameter Tuning & Model Improvement

In [14]:
def build_model(hidden_layers, neurons_per_layer, activation='relu', dropout_rate=0.2):
    model = models.Sequential()
    model.add(layers.Input(shape=(X_train_scaled.shape[1],)))
    
    for i in range(hidden_layers):
        model.add(layers.Dense(neurons_per_layer, activation=activation))
        if i < hidden_layers - 1:  # No dropout on last hidden layer
            model.add(layers.Dropout(dropout_rate))
    
    model.add(layers.Dense(1))
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Enhanced configurations to test
configs = [
    {'hidden_layers': 2, 'neurons': 64, 'dropout': 0.1},
    {'hidden_layers': 3, 'neurons': 128, 'dropout': 0.2},
    {'hidden_layers': 4, 'neurons': 64, 'dropout': 0.2},
    {'hidden_layers': 3, 'neurons': 256, 'dropout': 0.3},
    {'hidden_layers': 2, 'neurons': 128, 'dropout': 0.1}
]

tuning_results = []
histories = []

for i, config in enumerate(configs):
    print(f"\nTesting config {i+1}/{len(configs)}: {config}")
    model = build_model(config['hidden_layers'], config['neurons'], dropout_rate=config['dropout'])
    
    history = model.fit(
        X_train_scaled, y_train,
        validation_split=0.15,
        epochs=30,
        batch_size=32,
        verbose=0,
        callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
    )
    
    y_pred = model.predict(X_test_scaled, verbose=0).flatten()
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    tuning_results.append({**config, 'test_rmse': rmse, 'test_r2': r2})
    histories.append(history)
    
    print(f"  RMSE: {rmse:.2f} Wh, R²: {r2:.3f}")

# Display tuning results
results_df = pd.DataFrame(tuning_results)
print("\n--- Hyperparameter Tuning Results ---")
print(results_df.sort_values('test_rmse'))

# Find best configuration
best_config = results_df.loc[results_df['test_rmse'].idxmin()]
print(f"\nBest configuration: {best_config.to_dict()}")


Testing config 1/5: {'hidden_layers': 2, 'neurons': 64, 'dropout': 0.1}
  RMSE: 85.84 Wh, R²: 0.264

Testing config 2/5: {'hidden_layers': 3, 'neurons': 128, 'dropout': 0.2}
  RMSE: 83.36 Wh, R²: 0.306

Testing config 3/5: {'hidden_layers': 4, 'neurons': 64, 'dropout': 0.2}
  RMSE: 83.26 Wh, R²: 0.307

Testing config 4/5: {'hidden_layers': 3, 'neurons': 256, 'dropout': 0.3}
  RMSE: 80.84 Wh, R²: 0.347

Testing config 5/5: {'hidden_layers': 2, 'neurons': 128, 'dropout': 0.1}
  RMSE: 84.58 Wh, R²: 0.285

--- Hyperparameter Tuning Results ---
   hidden_layers  neurons  dropout  test_rmse   test_r2
3              3      256      0.3  80.837837  0.346987
2              4       64      0.2  83.258085  0.307300
1              3      128      0.2  83.362815  0.305556
4              2      128      0.1  84.583640  0.285068
0              2       64      0.1  85.841990  0.263637

Best configuration: {'hidden_layers': 3.0, 'neurons': 256.0, 'dropout': 0.3, 'test_rmse': 80.83783680539082, 'test_r

## 5. Final Model & Evaluation

In [15]:
# --- 5. FINAL MODEL & EVALUATION ---
# Based on tuning results, define improved final architecture
final_model = models.Sequential([
    layers.Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

# Enhanced optimizer with weight decay
optimizer = keras.optimizers.Adam(
    learning_rate=0.001,
    beta_1=0.9,
    beta_2=0.999,
    weight_decay=0.0001
)

final_model.compile(
    optimizer=optimizer,
    loss='huber',  # More robust to outliers than MSE
    metrics=['mae', 'mse']
)

final_model.summary()

print("\nTraining final model...")
history_final = final_model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    verbose=1,
    callbacks=[
        callbacks.EarlyStopping(
            monitor='val_loss', 
            patience=15,  # Increased patience
            restore_best_weights=True,
            min_delta=0.001
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', 
            factor=0.5, 
            patience=7,  # Changed from 5
            min_lr=0.00001
        )
    ]
)

# Final evaluation
y_train_pred_final = final_model.predict(X_train_scaled).flatten()
y_test_pred_final = final_model.predict(X_test_scaled).flatten()

# Calculate comprehensive metrics
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

train_rmse_final = np.sqrt(mean_squared_error(y_train, y_train_pred_final))
test_rmse_final = np.sqrt(mean_squared_error(y_test, y_test_pred_final))
train_mae_final = mean_absolute_error(y_train, y_train_pred_final)
test_mae_final = mean_absolute_error(y_test, y_test_pred_final)
train_mape_final = mean_absolute_percentage_error(y_train, y_train_pred_final) * 100
test_mape_final = mean_absolute_percentage_error(y_test, y_test_pred_final) * 100
train_r2_final = r2_score(y_train, y_train_pred_final)
test_r2_final = r2_score(y_test, y_test_pred_final)

print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE")
print("="*60)
print(f"Training RMSE: {train_rmse_final:.2f} Wh | R²: {train_r2_final:.3f}")
print(f"Test RMSE:     {test_rmse_final:.2f} Wh | R²: {test_r2_final:.3f}")
print(f"Test MAE:      {test_mae_final:.2f} Wh | MAPE: {test_mape_final:.2f}%")
print(f"MAE/Mean Ratio: {test_mae_final/y_test.mean():.2%}")
print(f"Improvement over baseline: {(test_rmse_baseline - test_rmse_final)/test_rmse_baseline:.2%}")

# Save all model artifacts
import json
import pickle

# Save model
final_model.save('final_neural_network_model.h5')

# Save training history
with open('training_history.json', 'w') as f:
    json.dump(history_final.history, f)

# Save scaler
with open('feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature names
with open('feature_names.txt', 'w') as f:
    f.write('\n'.join(X.columns.tolist()))

print("\nModel artifacts saved:")
print("  - final_neural_network_model.h5")
print("  - training_history.json")
print("  - feature_scaler.pkl")
print("  - feature_names.txt")

c:\Users\jenso\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_76 (Dense)                │ (None, 256)            │         7,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_31 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_77 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_32 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_78 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_79 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_80 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,689 (198.00 KB)

 Trainable params: 50,689 (198.00 KB)

 Non-trainable params: 0 (0.00 B)


Training final model...
Epoch 1/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 50.2700 - mae: 50.7645 - mse: 11606.0049 - val_loss: 42.9055 - val_mae: 43.3985 - val_mse: 9784.5400 - learning_rate: 0.0010
Epoch 2/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 43.8497 - mae: 44.3434 - mse: 10330.4248 - val_loss: 41.8318 - val_mae: 42.3226 - val_mse: 9378.3037 - learning_rate: 0.0010
Epoch 3/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 43.3183 - mae: 43.8118 - mse: 10120.0225 - val_loss: 41.5281 - val_mae: 42.0201 - val_mse: 9365.7910 - learning_rate: 0.0010
Epoch 4/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 42.8759 - mae: 43.3688 - mse: 9958.3516 - val_loss: 41.0493 - val_mae: 41.5416 - val_mse: 9135.0342 - learning_rate: 0.0010
Epoch 5/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 42.4207 - mae: 42.9135 - mse: 9803.3281 - val_loss: 40.6571 - val_mae: 41.1486 - val_mse: 9106.8564 - learning_rate: 0.0010
Epoch 6/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms


FINAL MODEL PERFORMANCE
Training RMSE: 73.75 Wh | R²: 0.489
Test RMSE:     80.45 Wh | R²: 0.353
Test MAE:      32.11 Wh | MAPE: 24.89%
MAE/Mean Ratio: 33.21%
Improvement over baseline: 4.17%

Model artifacts saved:
  - final_neural_network_model.h5
  - training_history.json
  - feature_scaler.pkl
  - feature_names.txt
